## Word Segmentation

**Author:** Diego Besada

Modified from [Peter Norvig's how to do things with words](https://nbviewer.org/url/norvig.com/ipython/How%20to%20Do%20Things%20with%20Words.ipynb)

In [1]:
import requests, regex as re
from collections import Counter
from functools import cache, reduce

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
def tokenize(text):
    'List all the word tokens (consecutive letters) in a text. Normalize to lowercase.'
    return re.findall('[a-z]+', text.lower())

CORPUS = requests.get('https://norvig.com/big.txt').text
COUNTS = Counter(tokenize(CORPUS))

In [4]:
N = sum(COUNTS.values())

def Pword(word):
    'Probability of `word`.'
    return COUNTS[word] / N

def Pwords(words):
    'Probability of words, assuming each word is independent of the others.'
    return reduce(lambda x, y: x * Pword(y), words, 1)

Pwords(['this', 'is', 'a', 'test'])

2.983396332800731e-11

In [5]:
def splits(text, start=0, L=20):
    'Return a list of all (first, rest) pairs; start <= len(first) <= L.'
    return [(text[:i], text[i:]) 
            for i in range(start, min(len(text), L)+1)]

splits('word')
splits('reallylongtext', 1, 4)

[('', 'word'), ('w', 'ord'), ('wo', 'rd'), ('wor', 'd'), ('word', '')]

[('r', 'eallylongtext'),
 ('re', 'allylongtext'),
 ('rea', 'llylongtext'),
 ('real', 'lylongtext')]

In [7]:
@cache
def segment(text):
    'Return a list of words that is the most probable segmentation of text.'
    
    if not text: 
        return []

    candidates = ([first] + segment(rest) for (first, rest) in splits(text, 1))
    return max(candidates, key=Pwords)

segment('thisisatest')
segment('shesellsseashellsbytheseashore')

['this', 'is', 'a', 'test']

['she', 'sell', 's', 'sea', 'shells', 'by', 'the', 'seashore']

### Bigram model

The unigram segmenter ignores word order. To condition each word on the previous one we use Norvig's bigram counts (`count_2w.txt`) together with the unigram counts (`count_1w.txt`) to estimate $P(w_i \mid w_{i-1})$. When a bigram is unseen we back off to the unigram probability. We also switch the unigram model to the larger corpus counts, as in Norvig's Sect. 7.

In [ ]:
def load_counts(url):
    'Read "token<TAB>count" lines into a Counter.'
    counts = Counter()
    for line in requests.get(url).text.splitlines():
        if '\t' not in line:
            continue
        key, count = line.rsplit('\t', 1)
        counts[key] = int(count)
    return counts

UNIGRAMS = load_counts('https://norvig.com/ngrams/count_1w.txt')
BIGRAMS = load_counts('https://norvig.com/ngrams/count_2w.txt')

N1 = sum(UNIGRAMS.values())
N2 = sum(BIGRAMS.values())

def P1w(word):
    'Unigram probability of `word` from the large corpus.'
    return UNIGRAMS.get(word, 0) / N1

def P2w(bigram):
    'Bigram probability of `bigram` from the large corpus.'
    return BIGRAMS.get(bigram, 0) / N2

def Pwords(words):
    'Probability of words, assuming each word is independent of the others.'
    return reduce(lambda x, y: x * P1w(y), words, 1)

len(UNIGRAMS), len(BIGRAMS)

(333333, 286358)

The conditional probability of a word given the previous one is $P(w_{i-1}w_i) / P(w_{i-1})$. If the bigram was never seen we fall back to half its unigram probability, which avoids a zero estimate:

In [ ]:
def cPword(word, prev):
    'Conditional probability of `word` given `prev`, backing off to the unigram.'
    bigram = prev + ' ' + word
    if P2w(bigram) > 0 and P1w(prev) > 0:
        return P2w(bigram) / P1w(prev)
    return P1w(word) / 2

def Pwords2(words, prev='<S>'):
    'Probability of words, using bigram data, given the previous word.'
    return reduce(
        lambda acc, i: acc * cPword(words[i], prev if i == 0 else words[i - 1]),
        range(len(words)),
        1
    )

@cache
def segment2(text, prev='<S>'):
    'Return the most probable segmentation of text using bigram data.'
    if not text:
        return []
    candidates = ([first] + segment2(rest, first) for (first, rest) in splits(text, 1))
    return max(candidates, key=lambda words: Pwords2(words, prev))

segment2('thisisatest')

['this', 'is', 'a', 'test']

### Experiment

We take a long string of text, remove its punctuation and white spaces and lowercase it. We then compare the segmentation given by the unigram model (`segment`) and by the bigram model (`segment2`).

In [19]:
examples = [
    'I have a lot of work to do today',
    'She looked at me for a long time',
    'At least we tried our best',
    'Can we talk about it another day'
]

for sentence in examples:
    text = re.sub(r'[^a-z]', '', sentence.lower())
    print(text)
    print('unigram:', segment(text))
    print('bigram :', segment2(text))
    print()

ihavealotofworktodotoday
unigram: ['i', 'have', 'alot', 'of', 'work', 'to', 'do', 'today']
bigram : ['i', 'have', 'a', 'lot', 'of', 'work', 'to', 'do', 'today']

shelookedatmeforalongtime
unigram: ['she', 'looked', 'at', 'me', 'for', 'along', 'time']
bigram : ['she', 'looked', 'at', 'me', 'for', 'a', 'long', 'time']

atleastwetriedourbest
unigram: ['atleast', 'we', 'tried', 'our', 'best']
bigram : ['at', 'least', 'we', 'tried', 'our', 'best']

canwetalkaboutitanotherday
unigram: ['can', 'we', 'talkabout', 'it', 'another', 'day']
bigram : ['can', 'we', 'talk', 'about', 'it', 'another', 'day']



### Conclusion

The unigram segmenter maximizes $\prod_i P_1(w_i)$ and ignores word order, so it always prefers whichever single token is the most frequent. As a result it merges two adjacent words whenever their concatenation happens to be a common token, even when that reading is wrong.

The bigram segmenter maximizes $\prod_i P(w_i \mid w_{i-1})$. By conditioning each word on the previous one, it can tell that the intended two-word split is more probable in context, and it repairs the boundary.

The bigram model is not always better: on `smallandinsignificant` the unigram keeps the correct `insignificant`, whereas the bigram over-splits it into `in significant`. Finally, a word missing from the vocabulary receives probability zero, and neither model can recover it.